# بازبینی یک تشخیص
آخرین اجرای موفق آزمون باز می‌شود. اگر این دفتر را در مسیر `private/` باز کرده‌اید، خروجی تصویری فقط در همان مسیر خصوصی ذخیره می‌شود.

In [ ]:
import base64
import json
from pathlib import Path
from html import escape
from IPython.display import HTML, display

here = Path.cwd()
private = here if here.name == "private" else here / "private"
reports = sorted(private.glob("review-*/report.json"), key=lambda path: path.stat().st_mtime)
if not reports:
    raise FileNotFoundError("Run tools/acceptance.py before opening the notebook.")
report_path = reports[-1]  # Select a different entry here to inspect an earlier run.
report = json.loads(report_path.read_text())
event = report["event"]
print(report_path.parent.name, "frame", event["frame_index"], "PTS", event["pts"], event["time_base"])

In [ ]:
def png(name):
    return "data:image/png;base64," + base64.b64encode((report_path.parent / name).read_bytes()).decode()

x, y, w, h = event["bbox_xywh"]
width, height = event["frame_size"]
identity = escape(event["detection_id"])
# A native disclosure control sits exactly on the detected face.
display(HTML(f"""
<div style="position:relative;width:min(100%,{width}px)">
  <img src="{png('frame.png')}" alt="Selected frame" style="width:100%;display:block">
  <details><summary aria-label="Inspect face" title="Inspect face"
    style="position:absolute;left:{100*x/width}%;top:{100*y/height}%;
    width:{100*w/width}%;height:{100*h/height}%;outline:3px solid orange;cursor:pointer">+</summary>
    <p><code>{identity}</code></p><img src="{png('crop.png')}" alt="S3 crop">
    <p>Kafka {escape(report['topic'])} / {report['partition']} / {report['offsets']}</p>
    <pre>{escape(json.dumps(report['sql'],indent=2))}</pre>
  </details>
</div>"""))

In [ ]:
print("Logical rows after replay:", report["logical_rows_after_replay"])
print("S3 crop matches frame pixels:", report["crop_pixels_equal"])
print("Seconds to first SQL visibility:", round(report["seconds_to_first_visible"], 3))